[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C11_RAG_Retrieval_Course/05_retrieval_metrics/05_retrieval_metrics.ipynb)

# 05 · 检索指标（纯 numpy/pandas）

目标：把 **Precision@k / Recall@k、MRR、AP / MAP、DCG / nDCG** 全部从零实现，每个都用**手算值**对拍验证，再理解它们各自对什么敏感。

路线：P@k & R@k → MRR → AP & MAP → DCG & nDCG → pandas 多指标并排 → 指标敏感度 → ✏️ 练习 → 📖 答案 → 🧪 真实数据(SQuAD+BM25)胶囊。

> **约定（全程一致）**：名次 **1-indexed**；nDCG 的 gain = `2^rel − 1`，折损 = `1/log2(pos+1)`（pos 从 1 开始）。
> 心智模型：**指标 = 给排好序的检索结果打分；不同指标奖励不同行为，选错就优化错方向**。

## 1 · Precision@k 与 Recall@k（从零 + 手算对拍）

二元相关性下最朴素的一对。**P@k** = top-k 中相关的比例；**R@k** = 相关文档中进了 top-k 的比例。
分子相同（top-k 命中的相关数），分母不同（P 除以 k，R 除以总相关数）。两者**都不看名次**。

In [ ]:
import numpy as np
import pandas as pd
rng = np.random.default_rng(0)

def precision_at_k(ranked_rel, k):
    '''ranked_rel: 按排序的二元相关性数组(1=相关)。返回 P@k。'''
    topk = np.asarray(ranked_rel)[:k]
    return float(topk.sum()) / k

def recall_at_k(ranked_rel, k, total_relevant=None):
    '''R@k = top-k 命中相关数 / 全部相关数。'''
    ranked_rel = np.asarray(ranked_rel)
    total = ranked_rel.sum() if total_relevant is None else total_relevant
    if total == 0:
        return 0.0
    return float(ranked_rel[:k].sum()) / float(total)

# 手算例子：排序 [1,0,1,0,1]，总相关=3
rel = [1, 0, 1, 0, 1]
print('P@3 =', precision_at_k(rel, 3), '  (top-3 命中 2 个 / 3 = 0.667)')
print('R@3 =', recall_at_k(rel, 3),    '  (top-3 命中 2 个 / 总 3 = 0.667)')
assert abs(precision_at_k(rel, 3) - 2/3) < 1e-9
assert abs(recall_at_k(rel, 3) - 2/3) < 1e-9
# k 增大：recall 单调不降，precision 可能降
assert recall_at_k(rel, 5) >= recall_at_k(rel, 3), 'k 增大 recall 不降'
assert abs(recall_at_k(rel, 5) - 1.0) < 1e-9, 'top-5 收齐全部 3 个相关 -> R@5=1'
print('✅ P@k/R@k 从零实现正确（对拍手算值）；R@k 随 k 单调不降')

## 2 · MRR：第一个相关结果排多前

`MRR = (1/|Q|) Σ 1/rank₁`，`rank₁` 是第一个相关结果的名次（1-indexed）。
只看**第一个**相关项：排第 1 得 1.0，第 2 得 0.5，第 3 得 0.33……一个相关的都没有则贡献 0。

In [ ]:
def reciprocal_rank(ranked_rel):
    '''单个查询：返回 1/第一个相关项名次(1-indexed)；无相关则 0。'''
    ranked_rel = np.asarray(ranked_rel)
    hits = np.where(ranked_rel == 1)[0]
    if len(hits) == 0:
        return 0.0
    first_rank = hits[0] + 1          # 0-indexed -> 1-indexed
    return 1.0 / first_rank

def mean_reciprocal_rank(list_of_ranked_rel):
    return float(np.mean([reciprocal_rank(r) for r in list_of_ranked_rel]))

# 单查询：第一个相关在第 3 位 -> 1/3
assert abs(reciprocal_rank([0, 0, 1, 0]) - 1/3) < 1e-9
assert reciprocal_rank([0, 0, 0]) == 0.0, '无相关 -> 0'
# 多查询取平均：[第1位, 第2位, 无] -> (1 + 0.5 + 0)/3
queries = [[1, 0, 0], [0, 1, 0], [0, 0, 0]]
mrr = mean_reciprocal_rank(queries)
print('各查询 RR:', [round(reciprocal_rank(r), 3) for r in queries])
print('MRR =', round(mrr, 4), ' (= (1 + 0.5 + 0)/3)')
assert abs(mrr - (1 + 0.5 + 0) / 3) < 1e-9
# 把第一个相关项提前，MRR 上升
assert reciprocal_rank([1, 0, 1]) > reciprocal_rank([0, 1, 1]), '第一个相关项越前 RR 越高'
print('✅ MRR 从零实现正确；第一个相关项越靠前，分数越高')

## 3 · AP 与 MAP：每个相关项处的精确率平均

`AP = (1/|R|) Σ_{相关项位置 k} P@k`：遍历列表，**每遇到一个相关文档**就记下到此为止的 P@k，再平均（除以相关总数）。
`MAP` = 所有查询 AP 的均值。AP 同时奖励『相关项多』和『相关项靠前』。

In [ ]:
def average_precision(ranked_rel):
    '''AP = 在每个相关项位置的 P@k 的平均(除以总相关数)。'''
    ranked_rel = np.asarray(ranked_rel)
    total_rel = ranked_rel.sum()
    if total_rel == 0:
        return 0.0
    hits = 0
    precision_sum = 0.0
    for i, r in enumerate(ranked_rel):
        if r == 1:
            hits += 1
            precision_sum += hits / (i + 1)      # P@(i+1)
    return precision_sum / total_rel

def mean_average_precision(list_of_ranked_rel):
    return float(np.mean([average_precision(r) for r in list_of_ranked_rel]))

# 手算例子：[1,0,1,1,0,0]，总相关=3
#   位置1: 1/1, 位置3: 2/3, 位置4: 3/4 -> AP=(1+0.667+0.75)/3=0.806
ap = average_precision([1, 0, 1, 1, 0, 0])
print('AP([1,0,1,1,0,0]) =', round(ap, 4), ' (手算 0.8056)')
assert abs(ap - (1/1 + 2/3 + 3/4) / 3) < 1e-9
# 完美排序(相关全在最前) AP=1
assert abs(average_precision([1, 1, 1, 0, 0]) - 1.0) < 1e-9
# 把相关项整体提前，AP 上升
assert average_precision([1, 1, 0, 0]) > average_precision([0, 0, 1, 1])
# MAP
mp = mean_average_precision([[1, 0, 1, 1, 0, 0], [1, 1, 1, 0, 0]])
print('MAP =', round(mp, 4), ' (= (0.8056 + 1.0)/2)')
assert abs(mp - (ap + 1.0) / 2) < 1e-9
print('✅ AP/MAP 从零实现正确（对拍手算值）；相关项越多越前，AP 越高')

## 4 · DCG 与 nDCG：分级相关性的排序指标

`DCG@k = Σ_{i=1}^{k} (2^rel_i − 1) / log2(i+1)`（gain 指数放大高相关，折损按位置打折）。
`nDCG@k = DCG@k / IDCG@k`，IDCG = 把文档按相关性**降序**排的理想 DCG。完美排序 nDCG=1，全零特判为 0。

In [ ]:
def dcg_at_k(ranked_grades, k=None):
    '''DCG@k，gain=2^rel-1, 折损=1/log2(pos+1), pos 从 1 开始。'''
    g = np.asarray(ranked_grades, dtype=float)
    if k is not None:
        g = g[:k]
    gains = (2.0 ** g - 1.0)
    discounts = 1.0 / np.log2(np.arange(2, len(g) + 2))   # pos=1 -> log2(2), pos=2 -> log2(3)...
    return float((gains * discounts).sum())

def ndcg_at_k(ranked_grades, k=None):
    dcg = dcg_at_k(ranked_grades, k)
    ideal = dcg_at_k(sorted(ranked_grades, reverse=True), k)  # IDCG
    if ideal == 0:
        return 0.0
    return dcg / ideal

# 手算例子：分级 [3,2,3,0,1,2]
grades = [3, 2, 3, 0, 1, 2]
dcg = dcg_at_k(grades)
ndcg = ndcg_at_k(grades)
print('DCG  =', round(dcg, 4), ' (手算 13.8483)')
print('IDCG =', round(dcg_at_k(sorted(grades, reverse=True)), 4), ' (手算 14.5954)')
print('nDCG =', round(ndcg, 4), ' (手算 0.9488)')
assert abs(dcg - 13.848264) < 1e-4, 'DCG 对拍手算'
assert abs(ndcg - 0.948811) < 1e-4, 'nDCG 对拍手算'
# 完美排序 nDCG=1；全零特判=0；nDCG 总在 [0,1]
assert abs(ndcg_at_k([3, 2, 2, 1, 0]) - 1.0) < 1e-9, '降序排列 nDCG=1'
assert ndcg_at_k([0, 0, 0]) == 0.0, '全零特判为 0'
assert 0.0 <= ndcg <= 1.0
print('✅ DCG/nDCG 从零实现正确（对拍手算值）；完美排序=1，全零=0')

## 5 · 用 pandas 把多指标并排：哪个排序更好？

把几个不同质量的排序放在一起，用 pandas 同时算 P@5/R@5/MRR/MAP/nDCG，并排展示。
你会看到：**没有单一指标能完整刻画排序质量**，要看一组。

In [ ]:
# 三个排序(同一查询, 二元相关性也当分级用, 总相关=3, 且全部落在 top-5 内)
rankings = {
    '完美   ': [1, 1, 1, 0, 0, 0],   # 相关全在最前
    '中段   ': [0, 0, 1, 1, 1, 0],   # 相关挤在中段(但都在 top-5)
    '交错   ': [1, 0, 1, 0, 1, 0],   # 相关交错分布
}
rows = []
for name, rel in rankings.items():
    rows.append({
        '排序': name,
        'P@5': round(precision_at_k(rel, 5), 3),
        'R@5': round(recall_at_k(rel, 5), 3),
        'MRR': round(reciprocal_rank(rel), 3),
        'AP':  round(average_precision(rel), 3),
        'nDCG': round(ndcg_at_k(rel), 3),
    })
df = pd.DataFrame(rows)
print(df.to_string(index=False))
# 完美排序在所有指标上都该最高
perfect = df[df['排序'] == '完美   '].iloc[0]
for m in ['MRR', 'AP', 'nDCG']:
    assert perfect[m] == df[m].max(), f'完美排序应在 {m} 上最高'
# P@5/R@5 对『中段』和『完美』给同分(top-5 都命中3个) -> 暴露『不看名次』的盲区
assert df[df['排序']=='完美   ']['R@5'].iloc[0] == df[df['排序']=='中段   ']['R@5'].iloc[0]
assert df[df['排序']=='完美   ']['P@5'].iloc[0] == df[df['排序']=='中段   ']['P@5'].iloc[0]
print('\n✅ 完美排序在看名次的指标(MRR/AP/nDCG)上全胜；但 P@5/R@5 对完美与中段给同分(不看名次的盲区)')

## 6 · 指标敏感度：同样的相关文档，不同排列

拿**同一组**相关文档（含一个高相关 grade=3），只改它们的**排列**，看 MRR / MAP / nDCG 各自如何反应。
结论：**nDCG 对『把高相关排最前』最敏感，MRR 只盯第一个相关项**。

In [ ]:
def to_binary(grades):
    return [1 if g > 0 else 0 for g in grades]

# 同样的等级集合 {3,1,1}，两种排列
A = [3, 1, 1, 0, 0]      # 高相关(3)在最前
B = [1, 1, 3, 0, 0]      # 高相关(3)在第3位
print(f"{'排列':<6}{'MRR':>8}{'MAP':>8}{'nDCG':>8}")
for name, g in [('A', A), ('B', B)]:
    b = to_binary(g)
    print(f'{name:<6}{reciprocal_rank(b):>8.3f}{average_precision(b):>8.3f}{ndcg_at_k(g):>8.3f}')
# MRR: 两者第一个相关项都在第1位 -> MRR 相同(对调高相关位置不敏感)
assert abs(reciprocal_rank(to_binary(A)) - reciprocal_rank(to_binary(B))) < 1e-9, 'MRR 对高相关项位置不敏感'
# nDCG: A 把高相关(3)放最前 -> nDCG 更高(对此敏感)
assert ndcg_at_k(A) > ndcg_at_k(B), 'nDCG 奖励把高相关排最前'
print('\n✅ 同样的相关文档不同排列：MRR 看不出差别(只盯第一个)，nDCG 明确偏好高相关靠前')
print('   -> 选指标要让它对你在意的变化敏感！')

---
## ✏️ 练习 1：Precision@k 与 Recall@k

实现 `precision_recall_at_k(ranked_rel, k, total_relevant=None)`，返回 `(P@k, R@k)` 元组。
（R@k 的分母：若给了 `total_relevant` 就用它，否则用 `ranked_rel` 里相关项总数；总相关为 0 时 R@k 返回 0.0。）

In [ ]:
def precision_recall_at_k(ranked_rel, k, total_relevant=None):
    # TODO: 返回 (P@k, R@k)
    #   P@k = top-k 命中相关数 / k
    #   R@k = top-k 命中相关数 / 总相关数 (0 相关时返回 0.0)
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
p3, r3 = precision_recall_at_k([1, 0, 1, 0, 1], 3)
assert abs(p3 - 2/3) < 1e-9 and abs(r3 - 2/3) < 1e-9
# 指定总相关数(有相关文档没进列表的情形)
p2, r2 = precision_recall_at_k([1, 1, 0], 2, total_relevant=4)
assert abs(p2 - 1.0) < 1e-9 and abs(r2 - 2/4) < 1e-9
# 全不相关
p0, r0 = precision_recall_at_k([0, 0, 0], 2)
assert p0 == 0.0 and r0 == 0.0
print('✅ 练习 1 通过：P@k/R@k 正确(含指定总相关数与全零情形)')

## ✏️ 练习 2：MRR

实现 `mean_reciprocal_rank(list_of_ranked_rel)`：对每个查询取第一个相关项名次(1-indexed)的倒数，再对所有查询平均。无相关项的查询贡献 0。

In [ ]:
def mean_reciprocal_rank(list_of_ranked_rel):
    # TODO: 每个查询 1/第一个相关项名次(无相关则0)，再平均
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
# 第一个相关项分别在 第2位/第1位/无 -> (0.5 + 1 + 0)/3
mrr = mean_reciprocal_rank([[0, 1, 0], [1, 0, 1], [0, 0, 0]])
assert abs(mrr - (0.5 + 1.0 + 0.0) / 3) < 1e-9
# 单查询第4位 -> 0.25
assert abs(mean_reciprocal_rank([[0, 0, 0, 1]]) - 0.25) < 1e-9
print('✅ 练习 2 通过：MRR 正确')

## ✏️ 练习 3：AP 与 MAP

实现 `average_precision(ranked_rel)`（每个相关项位置的 P@k 平均，除以总相关数；0 相关返回 0）与 `mean_average_precision(list_of_ranked_rel)`。

In [ ]:
def average_precision(ranked_rel):
    # TODO: 遍历, 每遇相关项累加 P@(当前位置), 最后除以总相关数
    raise NotImplementedError

def mean_average_precision(list_of_ranked_rel):
    # TODO: 所有查询 average_precision 的均值
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
# [0,1,1] -> 位置2: 1/2, 位置3: 2/3 -> (0.5+0.667)/2
assert abs(average_precision([0, 1, 1]) - (1/2 + 2/3) / 2) < 1e-9
assert abs(average_precision([1, 1]) - 1.0) < 1e-9
assert average_precision([0, 0]) == 0.0
mp = mean_average_precision([[0, 1, 1], [1, 1]])
assert abs(mp - ((1/2 + 2/3) / 2 + 1.0) / 2) < 1e-9
print('✅ 练习 3 通过：AP/MAP 正确')

## ✏️ 练习 4：nDCG

实现 `ndcg_at_k(ranked_grades, k=None)`：gain=`2^rel−1`，折损=`1/log2(pos+1)`(pos 从 1)，除以理想排序(降序)的 IDCG；IDCG=0 时返回 0.0。

In [ ]:
def ndcg_at_k(ranked_grades, k=None):
    # TODO: 1) 算 DCG@k(gain=2^rel-1, 折损=1/log2(pos+1))
    #       2) 算 IDCG = 把 grades 降序排后的 DCG@k
    #       3) nDCG = DCG/IDCG (IDCG=0 -> 0.0)
    raise NotImplementedError

In [ ]:
# —— 练习 4 自测 ——
# 手算 [3,2,3,0,1,2] -> nDCG=0.948811
assert abs(ndcg_at_k([3, 2, 3, 0, 1, 2]) - 0.948811) < 1e-4
# 降序排列 -> 1.0；全零 -> 0.0；范围 [0,1]
assert abs(ndcg_at_k([3, 2, 1]) - 1.0) < 1e-9
assert ndcg_at_k([0, 0]) == 0.0
# 截断 k：只看前 2 个
v = ndcg_at_k([1, 3, 0, 0], k=2)
assert 0.0 <= v <= 1.0
print('✅ 练习 4 通过：nDCG 正确(对拍手算值, 含完美/全零/截断)')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def precision_recall_at_k(ranked_rel, k, total_relevant=None):
    arr = np.asarray(ranked_rel)
    hit = float(arr[:k].sum())
    total = arr.sum() if total_relevant is None else total_relevant
    pk = hit / k
    rk = 0.0 if total == 0 else hit / float(total)
    return pk, rk

In [ ]:
# 练习 2 参考答案
def mean_reciprocal_rank(list_of_ranked_rel):
    rrs = []
    for r in list_of_ranked_rel:
        hits = np.where(np.asarray(r) == 1)[0]
        rrs.append(0.0 if len(hits) == 0 else 1.0 / (hits[0] + 1))
    return float(np.mean(rrs))

In [ ]:
# 练习 3 参考答案
def average_precision(ranked_rel):
    arr = np.asarray(ranked_rel)
    total = arr.sum()
    if total == 0:
        return 0.0
    hits, psum = 0, 0.0
    for i, r in enumerate(arr):
        if r == 1:
            hits += 1
            psum += hits / (i + 1)
    return psum / total

def mean_average_precision(list_of_ranked_rel):
    return float(np.mean([average_precision(r) for r in list_of_ranked_rel]))

In [ ]:
# 练习 4 参考答案
def ndcg_at_k(ranked_grades, k=None):
    def dcg(gs):
        gs = np.asarray(gs, dtype=float)
        if k is not None:
            gs = gs[:k]
        gains = 2.0 ** gs - 1.0
        disc = 1.0 / np.log2(np.arange(2, len(gs) + 2))
        return float((gains * disc).sum())
    ideal = dcg(sorted(ranked_grades, reverse=True))
    return 0.0 if ideal == 0 else dcg(ranked_grades) / ideal

---
## 🧪 真实数据胶囊：在 SQuAD 上用 BM25 检索并算 MRR/MAP/nDCG

用真实 **SQuAD**：把段落当语料、问题当查询，用 **BM25** 检索，相关性 = 『该段落是否是这个问题的标准答案段落』（二元）。
优先**联网下载**，失败**回退到内置真实 SQuAD 样本**。然后对所有查询算 MRR / MAP / nDCG。

In [ ]:
import math
from collections import Counter

# 优先联网拉真实 SQuAD；失败回退内置真实样本(每段落取1问题, 各属不同段落)
def load_squad_samples(n=6):
    try:
        import urllib.request, json
        url = 'https://rajpurkar.github.io/SQuAD-explorer/dataset/dev-v2.0.json'
        with urllib.request.urlopen(url, timeout=5) as f:
            data = json.load(f)
        out = []
        for art in data['data']:
            for para in art['paragraphs']:
                ctx = para['context']
                for qa in para['qas']:
                    if not qa.get('is_impossible', False) and qa['answers']:
                        out.append((qa['question'], ctx, qa['answers'][0]['text']))
                        break
                if len(out) >= n: break
            if len(out) >= n: break
        print(f'✅ 联网加载 SQuAD 成功，取 {len(out)} 条(各属不同段落)')
        return out
    except Exception as e:
        print(f'⚠ 联网失败({type(e).__name__})，回退内置真实 SQuAD 样本')
        return [
            ('In what country is Normandy located?',
             'The Normans were the people who in the 10th and 11th centuries gave their '
             'name to Normandy, a region in France. They were descended from Norse raiders.',
             'France'),
            ('When were the Normans in Normandy?',
             'The Normans were the people who in the 10th and 11th centuries gave their '
             'name to Normandy, a region in France.',
             '10th and 11th centuries'),
            ('What is the capital of France?',
             'Paris is the capital and most populous city of France, situated on the Seine.',
             'Paris'),
            ('What language did the Normans speak?',
             'The Norman dynasty had a major political and cultural impact; they spoke a '
             'language that evolved into Norman French.',
             'Norman French'),
            ('Photosynthesis occurs in which organelle?',
             'In plants, photosynthesis takes place in chloroplasts, which contain the '
             'pigment chlorophyll that captures light energy.',
             'chloroplasts'),
            ('What gas do plants absorb during photosynthesis?',
             'During photosynthesis plants absorb carbon dioxide from the air and release '
             'oxygen as a byproduct.',
             'carbon dioxide'),
        ]

class BM25:
    def __init__(self, corpus, k1=1.5, b=0.75):
        self.k1, self.b = k1, b
        self.docs = [d.lower().split() for d in corpus]
        self.N = len(self.docs)
        self.doc_len = [len(d) for d in self.docs]
        self.avgdl = sum(self.doc_len) / self.N
        df = Counter()
        for d in self.docs:
            for t in set(d):
                df[t] += 1
        self.idf = {t: math.log(1 + (self.N - n + 0.5) / (n + 0.5)) for t, n in df.items()}
        self.tf = [Counter(d) for d in self.docs]
    def rank(self, query):
        scores = []
        for i in range(self.N):
            s, dl = 0.0, self.doc_len[i]
            for t in query.lower().split():
                if t in self.idf:
                    tf = self.tf[i][t]
                    s += self.idf[t] * tf * (self.k1 + 1) / (tf + self.k1 * (1 - self.b + self.b * dl / self.avgdl))
            scores.append(s)
        return np.argsort(-np.array(scores))      # 文档下标按 BM25 降序

samples = load_squad_samples(6)
questions = [q for q, _, _ in samples]
contexts = [c for _, c, _ in samples]
bm = BM25(contexts)
# 对每个问题 i，正确(相关)段落是 contexts[i]；把 BM25 排序转成二元相关性序列
ranked_rels = []
for i, q in enumerate(questions):
    order = bm.rank(q)
    ranked_rels.append([1 if doc == i else 0 for doc in order])
print(f'\n语料 {len(contexts)} 段, 查询 {len(questions)} 个')
print('MRR =', round(mean_reciprocal_rank(ranked_rels), 4))
print('MAP =', round(mean_average_precision(ranked_rels), 4))
print('平均 nDCG =', round(float(np.mean([ndcg_at_k(r) for r in ranked_rels])), 4))

**🧪 胶囊练习**：实现 `evaluate_all(ranked_rels)`：对一批查询的二元相关性序列，返回 `{'MRR':..., 'MAP':..., 'nDCG':...}`（nDCG 取各查询平均）。这是检索评测的『一键体检』。

In [ ]:
def evaluate_all(ranked_rels):
    # TODO: 返回 dict(MRR=..., MAP=..., nDCG=...)，复用上面实现的指标函数
    #   nDCG 用各查询 ndcg_at_k 的平均
    raise NotImplementedError

In [ ]:
# 自测
report = evaluate_all(ranked_rels)
print('检索评测报告:', {k: round(v, 4) for k, v in report.items()})
assert set(report) == {'MRR', 'MAP', 'nDCG'}
for v in report.values():
    assert 0.0 <= v <= 1.0
# BM25 在 SQuAD 上(问题与答案段落词重叠高)应远好于随机：MRR 明显 > 0
assert report['MRR'] > 1.0 / len(contexts), 'BM25 应显著优于随机基线'
print('✅ 胶囊练习通过：在真实 SQuAD 上一键算出 MRR/MAP/nDCG，均显著优于随机')

In [ ]:
# 📖 胶囊参考答案
def evaluate_all(ranked_rels):
    return {
        'MRR': mean_reciprocal_rank(ranked_rels),
        'MAP': mean_average_precision(ranked_rels),
        'nDCG': float(np.mean([ndcg_at_k(r) for r in ranked_rels])),
    }

### 小结
- 检索输出是**有序列表**，评它需要 **qrels**(人工相关性标注)；二元标注用 P/R/MRR/MAP，分级标注用 nDCG。
- **P@k/R@k**：分子同(top-k命中相关数)、分母异(÷k vs ÷总相关)；**都不看名次**，是最大盲区。
- **MRR**：只看第一个相关项名次的倒数，适合『找一个答案』；对后续相关项完全无视。
- **AP/MAP**：每个相关项处 P@k 的平均，奖励『相关项多且靠前』，二元下最全面。
- **nDCG**：gain(2^rel-1) × 折损(1/log2(pos+1)) ÷ IDCG，唯一能用**分级相关性**，最贴近用户体验。
- **选指标=用对的尺子**；更要**报一组互补指标 + 看分布 + 做显著性检验**，别被单一均值骗了。

下一站：**模块 06 · 长上下文评测** —— 当上下文窗口长到能装整本书，检索还需不需要？NIAH 大海捞针与 lost-in-the-middle 位置偏置。